# 01｜Tokenizer 与 Token

目标：理解自然语言如何转换为模型能够处理的整数，以及如何还原为文本。

## 1 BPE算法
Byte-Pair Encoding（字节对编码）

### 1.1 算法核心逻辑
BPE的训练过程本质上是一个迭代的数据压缩过程。

- 1.初始化：将所有文本拆解为基础单元（通常是字节）。
- 2.统计：统计所有相邻单元对（Pair）在数据中出现的频率。
- 3.合并：找到频率最高的那个对（例如 ('e', 's')），将其合并为一个新的符号（'es'），并分配一个新的ID。
- 4.迭代：重复步骤2和3，直到达到预设的词表大小（Vocabulary Size）。

我们将代码分为三个核心功能模块进行讲解：统计（Statistics）、合并（Merge） 和 训练主循环（Training Loop）。

### 1.1 BPE代码详解

#### 1.1.1 代码详解：统计频率 (get_stats)
对当前的token序列统计每个相邻token对的出现次数

In [28]:
def get_stats(ids):  
    """  
    输入:  
    ids (list of integers): 当前的Token ID列表。  
    输出:  
    counts (dict): 映射 (id1, id2) -> frequency  
    """  
    counts = {}  
    # zip(ids, ids[1:]) 是一个Python技巧    
    # zip后生成: [(1, 2), (2, 3), (3, 4)]  
    # 这正是我们需要统计的所有相邻对  
    for pair in zip(ids, ids[1:]):   
        counts[pair] = counts.get(pair, 0) + 1  
    return counts

深入解析与知识点：

- 输入数据的本质：这里的 ids 最初是什么？在现代LLM（如GPT-4）中，我们使用字节级（Byte-level）BPE。这意味着 ids 的初始状态是文本UTF-8编码后的字节序列，范围在0-255之间。例如，英文字母 a 是97，中文 你 是三个字节。这种设计极其重要，因为它保证了Tokenizer可以处理任何Unicode字符串，哪怕是Emoji或从未见过的外文，因为一切皆为字节。

- 时间复杂度：此函数的时间复杂度为 O(N)，其中 N 是序列长度。在训练过程中，每次合并都需要重新扫描整个语料库，这使得朴素实现的BPE训练非常慢（$O(N cdot V)$，其中 
V 是合并次数）。工业级实现（如Rust编写的Tiktoken）会使用链表或优先队列来优化更新过程，避免全量扫描

#### 1.1.2 代码详解：执行合并 (merge)
一旦我们找到了出现频率最高的对（比如 (101, 115) 对应 ('e', 's')），我们就需要将序列中所有的 (101, 115) 替换为新的Token ID

In [25]:
def merge(ids, pair, idx):  
    """  
    输入:  
    ids (list): 当前的Token列表  
    pair (tuple): 要合并的一对Token, 例如 (101, 115)  
    idx (int): 分配给这个新Token的ID, 例如 256  
      

    输出:  
    newids (list): 合并后的新列表  
    """  
    newids = []  
    i = 0  
    while i < len(ids):  
        #检查是否刚好碰到了我们要合并的pair，且没有越界  
        #注意: i < len(ids) - 1 是为了防止检查 ids[i+1] 时越界  
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:  
            newids.append(idx) # 替换为新的ID  
            i += 2 # 跳过接下来两个元素，因为它们已经被合并了  
        else:  
            newids.append(ids[i]) # 保持原样  
            i += 1  
    return newids

以上代码所做的事：对token列表循环，寻找要合并的token对pair，找到之后将原token对合并为新的一个token，并分配新的token ID：idx，同时更新token列表，得到newids

深入解析与知识点：

- 贪婪算法（Greedy Strategy）：BPE是贪婪的。只要选定了频率最高的Pair，它就会在全局范围内将所有的实例都替换掉。这不同于某些动态规划算法。
- 序列变短：每次合并操作都会使 ids 列表的长度变短。这正是“压缩”的体现。对于大模型，更短的序列意味着能放入更多的上下文信息。
- 新ID的分配：基础词表是0-255。第一次合并产生的Token ID是256，第二次是257，依此类推。GPT-4的 cl100k_base 词表大小约为100,277，这意味着这个合并过程在训练阶段重复了约10万次。

**补充说明：**

```
  字符
    ↓
  你

  Unicode 码点
    ↓
  U+4F60

  使用 UTF-8 编码
    ↓
  E4 BD A0

  转换为十进制字节
    ↓
  [228, 189, 160]
```
所以完整过程应该是：
```
文本字符
  → Unicode 码点
  → UTF-8 字节
  → Byte-level BPE token
  → 输入模型的 token ID
```

#### 1.1.3 代码详解：训练主循环 (train)
将上述两个函数结合，就构成了BPE的训练器。

In [ ]:
def train(text, vocab_size, verbose=False):  
    """  
    输入:  
    text (str): 训练语料文本  
    vocab_size (int): 目标词表大小 (例如 50257)  
    """  
    assert vocab_size >= 256  
    num_merges = vocab_size - 256 # 需要进行的合并次数  
      

    # 1. 预处理：将文本转换为UTF-8字节流  
    text_bytes = text.encode("utf-8")  
    ids = list(text_bytes) # 初始列表，元素范围 0-255  
      
    merges = {} # 记录合并规则: (p0, p1) -> idx  
      
    print(f"Original length: {len(ids)}")  
      
    for i in range(num_merges):  
        # 2. 统计当前频率  
        stats = get_stats(ids)  # 得到每一个pair及其出现次数
        if not stats:  
            break # 如果没有可以合并的对，提前退出  
              
        # 3. 找到频率最高的对  
        # key=stats.get 表示按字典的值(频率)排序  
        pair = max(stats, key=stats.get)  
          
        # 4. 分配新ID  
        idx = 256 + i  # 新ID从小到大进行分配，越先进行合并的pair，分配的ID越小
          
        # 5. 执行合并  
        ids = merge(ids, pair, idx)  
          
        # 6. 记录规则  
        merges[pair] = idx  # merges是对每轮合并的pair及其对应的新ID的记录
          
        if verbose:  
            print(f"Merge {i+1}/{num_merges}: {pair} -> {idx} (count: {stats[pair]})")  
              
    return merges

深入解析与知识点：

- 核心产物：训练结束后，最重要的产物不是 ids，而是 merges 字典。这个字典定义了Tokenizer的“知识”。当我们下载一个预训练模型时，tokenizer.json 或 merges.txt 存储的就是这个字典。
- 确定性与Tie-breaking：如果两个Pair频率相同怎么办？Python的 max 函数在值相同时会返回第一个遇到的键。为了保证Tokenizer的可复现性（Deterministic），在工业级实现中通常会规定：当频率相同时，按照Pair中字符的字典序（Lexicographical order）进行选择。
- 词表大小的权衡：vocab_size 是一个关键超参数。
    - 太小：导致序列过长，模型推理慢，无法捕捉长距离依赖。
    - 太大：导致Embedding矩阵（vocab_size x hidden_dim）参数量激增，增加训练负担；且由于稀有词频次太低，其Embedding可能训练不充分（Undertrained）。

通过以上train函数会得到训练好的merges规则，也就是初始的256词表中的哪些pair会被合并，以及其合并后的新ID（同时对应合并次序）

这个合并规则会在推理时被直接调用

**推理时不会根据当前text生成新的merges规则**

#### 1.1.4 代码详解：推理阶段的编码 (encode)
有了训练好的 merges 规则，如何将新的文本转化为Token IDs？这是一个极易出错的环节。初学者常犯的错误是：在推理时依然每次找文本中频率最高的对。这是错误的。推理时，必须严格按照训练时确定的优先级顺序进行合并。

In [ ]:
def encode(text, merges):  
    # 1. 将初始文本转为字节流  
    ids = list(text.encode("utf-8"))  
      

    while len(ids) >= 2:  
        # 获取当前文本中所有相邻对  
        stats = get_stats(ids)  
          
        # 寻找“在merges规则表中存在，且ID最小（即最早被训练出来）”的对  
        # 因为ID越小，说明它在训练集中频率越高，优先级越高  
        pair_to_merge = None  
        min_rank = float("inf") # rank即ID  
          
        # 对初始ID pair进行遍历，找到最小的rank（pair ID），也就是第一个进行合并的pair
        for pair in stats:  
            if pair in merges:  
                rank = merges[pair]  
                if rank < min_rank:  
                    min_rank = rank  
                    pair_to_merge = pair  
          
        # 如果当前序列中没有任何对在我们的规则表中，停止  
        if pair_to_merge is None:  
            break  
              
        # 执行合并  
        ids = merge(ids, pair_to_merge, min_rank)  

        #一直进行合并至text（ids）中没有merges中的pair，或序列只剩一个词为止
          
    return ids

深入解析与知识点：

- 优先级逻辑：代码中的 min_rank 逻辑至关重要。假设我们有规则 (a, b) -> X (Rank 1) 和 (b, c) -> Y (Rank 2)。对于输入 abc，我们必须先合并Rank 1的规则，得到 Xc。如果我们先合并 bc 得到 aY，就破坏了BPE的构造一致性，导致生成的Token序列与训练时分布不一致。
**也就是说，需要严格按照训练时生成的merges规则进行pair的合并**
- 递归与迭代：上述代码使用了类似迭代的方式。在Python的 minbpe 实现中，也可以通过递归来实现，或者使用更高效的Regex拆分后独立处理每个块（详见后文GPT-2/4部分）。

#### 1.1.5 代码详解：解码 (decode)
解码是编码的逆过程，相对简单，但有一个关键细节：处理无效字节。

In [ ]:
def decode(ids, vocab):  
    """  
    ids: token ID列表  
    vocab: 映射 {idx: bytes} (由基础字节和merges反推得到)  
    """  
    # 将所有ID映射回字节串，并拼接  
    tokens = b"".join(vocab[idx] for idx in ids)  
    #将ids中的所有token ID映射回对应的字节串（0-255），并拼接成一个完整的字节流
      

    # errors="replace" 是关键  
    text = tokens.decode("utf-8", errors="replace")  
    return text

深入解析与知识点：

- errors="replace"：在GPT-3或GPT-4的输出中，我们偶尔会看到 (Replacement Character)。这是因为大模型有时会生成不完整的Token序列。例如，一个中文字符由3个字节组成，如果模型只输出了前2个字节就停止了（比如因为max_tokens限制），那么这2个字节无法构成合法的UTF-8字符。使用 replace 策略可以防止程序崩溃。

replace：

补充：UTF-8 的规则
```
0xxxxxxx → 单字节字符
110xxxxx → 两字节字符的开头
1110xxxx → 三字节字符的开头
11110xxx → 四字节字符的开头
10xxxxxx → 后续字节
```
所以，当解码器读到3字节的第一个字节时，就会知道这个字符应该由三个字节组成，如果模型只输出了前2个字节就停止了，errors="replace"：输出 �，表示这里存在损坏或缺失的内容。

## 2 算法三巨头深度对比：BPE, WordPiece 与 Unigram

### 2.1 WordPiece：从频率到概率的跨越
* **BPE标准**：选择 **频次（Frequency）** 最高的对。  
  * 目标：最大化数据压缩比。  
* **WordPiece标准**：选择合并后能使训练数据**似然度（Likelihood）** 增加最多的对。  
  
  * 这等价于选择**点互信息（Pointwise Mutual Information, PMI）** 最高的对。  
  * WordPiece 得分公式：
  
    $$text{Score}(A, B) = \frac{P(AB)}{P(A) \times P(B)}$$
  
    其中 $P(AB)$ 是对 $AB$ 出现的概率，$P(A)$ 和 $P(B)$ 是各自的概率。

### 2.2 Unigram：基于概率图模型的自顶向下剪枝**

Unigram Tokenization（主要在SentencePiece库中实现）的思路与前两者截然相反。BPE和WordPiece是**自底向上（Bottom-up）** 的构造，Unigram是**自顶向下（Top-down）** 的。

## 3 GPT分词器的演进
OpenAI的GPT系列一直沿用BPE算法，但在细节上进行了极其关键的优化。了解这些演进，能让我们明白为什么GPT-4在写代码和处理非英语语言时比GPT-2强得多。

### 3.1 预分词
BPE算法本身是“盲目”的。如果不加干预，它可能会跨越标点和单词的边界进行合并。
>例如，它可能会把句子结尾的 "dog." 中的 "g." 合并为一个Token。这通常是不希望看到的，因为标点符号通常有独立的语法意义。

因此，在运行BPE合并之前，需要先用**正则表达式（Regex）** 将文本切分为一个个基础的“单词块”。BPE只能在这些块内部进行合并，不能跨块。

### 3.2 字节级BPE的精妙实现 (bytes_to_unicode)

在 GPT-2 的源码中，`bytes_to_unicode` 是一个极具迷惑性但至关重要的函数。它是 BPE 算法能够处理**任意字节流**（包括 Unicode 乱码、二进制文件、Emoji 等）的基石。

## 4 特殊token的处理

维护一个独立的字典来存储特殊Token。在分词开始前，它会先用正则把这些特殊字符串“抠”出来，不参与BPE合并，直接赋予特定的ID。

In [29]:
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'model').exists() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('./model')
print(tokenizer)

PreTrainedTokenizerFast(name_or_path='./model', vocab_size=6400, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|im_start|>', 'eos_token': '<|im_end|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>', '<|audio_start|>', '<|audio_end|>', '<|audio_pad|>', '<tts_pad>', '<tts_text_bos>', '<tts_text_eod>', '<tts_text_bos_single>']}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<|im_end|>", rstrip=False, l

In [27]:
text = '你好，MiniMind！'
token_ids = tokenizer.encode(text)
tokens = tokenizer.convert_ids_to_tokens(token_ids)

print('原文：', text)
print('Token ID：', token_ids)
print('Token：', tokens)
print('解码：', tokenizer.decode(token_ids))
print('Token 数量：', len(token_ids))

原文： 你好，MiniMind！
Token ID： [1968, 294, 80, 301, 108, 80, 916, 1364]
Token： ['ä½łå¥½', 'ï¼Į', 'M', 'in', 'i', 'M', 'ind', 'ï¼ģ']
解码： 你好，MiniMind！
Token 数量： 8


### 为什么 Token 看起来像乱码？

MiniMind 使用 ByteLevel BPE。`convert_ids_to_tokens()` 展示的是分词器内部的字节表示，中文可能看起来像乱码；只要 `decode()` 能还原原文，就不表示数据损坏。

## 观察特殊 Token

BOS 表示序列开始，EOS 表示序列结束，PAD 用于把不同长度的样本补成相同长度。

In [13]:
special_tokens = {
    'BOS': (tokenizer.bos_token, tokenizer.bos_token_id),
    'EOS': (tokenizer.eos_token, tokenizer.eos_token_id),
    'PAD': (tokenizer.pad_token, tokenizer.pad_token_id),
}
special_tokens

{'BOS': ('<|im_start|>', 1),
 'EOS': ('<|im_end|>', 2),
 'PAD': ('<|endoftext|>', 0)}

## 对话模板

对话模型不仅需要文本，还要区分 system、user 和 assistant 的角色。

In [ ]:
messages = [
    {'role': 'system', 'content': '你是一个乐于助人的助手。'},
    {'role': 'user', 'content': '请解释什么是 token。'},
]
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
print(prompt)

## 我的学习记录

- Token 和汉字/单词是否一一对应？
- BOS、EOS、PAD 分别有什么作用？
- 为什么同一句话编码后可以恢复原文？
- 我修改文本后观察到的变化：

## Minimind的tokenizer实现

注意区分：
- tokenizer.json = 一本新华字典
- tokenizers     = 能够查阅和编写这本字典的程序
- AutoTokenizer  = 对这个程序进行高级封装的加载器

#### 1. tokenizer.json
本地文件：
[tokenizer.json](/Users/shelly/Desktop/minimind/minimind/model/tokenizer.json)

里面保存：
- Token 词表：Token 与 Token ID 的对应关系。
- BPE 合并规则：哪些字符或字节组合应该合并成一个 Token。
- ByteLevel 预处理规则。
- 解码规则：如何从 Token ID 还原文字。
- Added Token：后来额外加入的 <think>、<tool_call> 等标记。
- 特殊 Token 的基本信息。
它是数据文件，自己不能执行。

可以把它理解成：
tokenizer.json = 词典正文 + 分词算法规则

#### 2. tokenizers
这是一个 Python 包：
`from tokenizers import Tokenizer`

它提供底层功能：
- 训练 BPE 词表；
- 加载 tokenizer.json；
- 把文字编码成 Token ID；
- 把 Token ID 解码成文字。

#### 3. AutoTokenizer
是 transformers 包提供的高级封装：
它不仅会加载 `tokenizer.json`，还会读取：
`tokenizer_config.json`

它们之间的关系可以表示为：
```text
transformers.AutoTokenizer
          ↓ 高级封装
      tokenizers 库
          ↓ 读取
tokenizer.json + tokenizer_config.json
```

补充：
#### 4.tokenizer_config.json：如何使用这本词典
[tokenizer_config.json](/Users/shelly/Desktop/minimind/minimind/model/tokenizer_config.json) 不负责训练 BPE 词表，而是告诉 Transformers 应该怎样使用 tokenizer.json。
它主要保存：
- 哪个 Token 是 BOS。
- 哪个 Token 是 EOS。
- 哪个 Token 是 PAD。
- 哪个 Token 是 UNK。
- 最大长度 model_max_length。
- 是否自动添加 BOS/EOS。
- 使用哪个 Tokenizer 类。
- 对话模板 chat_template。
- 工具调用、思考、多模态标记的使用方式。

例如：
```json
{
    "bos_token": "<|im_start|>",
    "eos_token": "<|im_end|>",
    "pad_token": "<|endoftext|>",
    "unk_token": "<|endoftext|>",
    "tokenizer_class": "PreTrainedTokenizerFast"
}
```
可以把它理解成：
tokenizer_config.json = 词典使用说明书

**tokenizer.json 和 tokenizer_config.json**
- tokenizer.json：主要负责“文本 ↔ Token ID”的双向转换。
- tokenizer_config.json：主要规定“如何组织和使用这些 Token”，并不是模型读取、输出 Token 的核心程序。

```text
原始对话 messages
        ↓
tokenizer_config.json 的 chat_template
        ↓
带角色标记的字符串 prompt
        ↓
tokenizer.json
        ↓
Token ID
        ↓
MiniMind 模型
        ↓
预测下一个 Token ID
        ↓
tokenizer.json
        ↓
输出文字
```

### Minimind的tokenizer训练
`train_tokenizer.py`
这个脚本的目的不是使用现有词表，而是重新训练一个新词表：
```python
tokenizer = Tokenizer(models.BPE())
```

⚠️为了正常使用 MiniMind：不应该，也不需要运行以下代码

`train_tokenizer.py` 只是帮助你学习“词典是怎么训练出来的”。它生成的新词表与官方模型权重不兼容。

代码流程是：
```text
读取 SFT 对话数据
→ 提取所有消息正文
→ 用 ByteLevel 预处理文本
→ 训练 BPE 词表
→ 加入特殊 Token
→ 保存 tokenizer.json
→ 生成 tokenizer_config.json
→ 加载并测试新分词器
```
注意，这段代码是“重新训练一本新词典”，不是使用 MiniMind 已有分词器。

MiniMind 已有分词器位于：

[tokenizer.json](/Users/shelly/Desktop/minimind/minimind/model/tokenizer.json)

[tokenizer_config.json](/Users/shelly/Desktop/minimind/minimind/model/tokenizer_config.json)

仓库 README 明确说明：不建议重新训练 Tokenizer，因为新词表中的 Token ID 会发生变化，原有模型权重将不能正常使用。

#### 1 导入和配置

In [ ]:
# 注：不建议再重复训练tokenizer（“词典”），MiniMind已自带，此脚本仅供学习和参考。基于不同词典训练的模型将导致输出完全不统一，降低社区的模型复用性
# Note: It is not recommended to re-train the tokenizer. MiniMind already includes one. This script is for learning and reference only. Training models with different tokenizers will lead to inconsistent outputs and reduce model reusability in the community.
import os
import json
from tokenizers import decoders, models, pre_tokenizers, trainers, Tokenizer
# decoders：把 Token ID 解码回文字
# models：分词模型，例如 BPE。
# pre_tokenizers：正式分词前的预处理。
# trainers：负责从语料统计和训练词表。
# Tokenizer：分词器主体。

DATA_PATH = '../dataset/sft_t2t_mini.jsonl'
TOKENIZER_DIR = '../model_learn_tokenizer/'
# 新训练的分词器保存到：../model_learn_tokenizer/
VOCAB_SIZE = 6400
# 目标词表大小为 6400
SPECIAL_TOKENS_NUM = 36
# 计划保留 36 个特殊 Token 位置。

#### 2 读取训练文本

In [ ]:
#读取训练文本
def get_texts(data_path):
    with open(data_path, 'r', encoding='utf-8', errors='ignore') as f:
    #打开数据文件：'r'：只读。utf-8：按 UTF-8 解码。
    #errors='ignore'：遇到无法解码的字符时跳过，而不是中止。
        for i, line in enumerate(f):
        #line 是一整行 JSON。i 是行号，从 0 开始。
            if i >= 10000: break # 选10000行测试
            try: #尝试解析当前行。某一行损坏时，避免整个训练终止。
                data = json.loads(line) #把 JSON 字符串转成 Python 字典。
                contents = [item.get('content') for item in data.get('conversations', []) if item.get('content')]
                if contents: #如果找到了内容，就用换行连接：
                    yield "\n".join(contents)
            except json.JSONDecodeError: #如果某一行不是合法 JSON，就跳过这一行。
                continue

其中，try中contents的操作是：

假设一行数据是：

In [ ]:
{
  "conversations": [
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好！"}
  ]
}

解析后 data 就是对应的 Python 字典。

In [ ]:
contents = [
    item.get('content')
    for item in data.get('conversations', [])
    if item.get('content')
]

#这是列表推导式，可以拆成：

contents = []

for item in data.get('conversations', []):
    if item.get('content'):
        contents.append(item.get('content'))

作用是：
- 取得 conversations 列表。
- 遍历其中每条消息。
- 提取非空的 content。
- 不关心消息是 user 还是 assistant。

#### 3 训练 BPE 分词器

定义训练函数，接收：
- 数据文件路径；
- 输出目录；
- 词表大小；
- 特殊 Token 总数。

In [ ]:

def train_tokenizer(
    data_path, 
    tokenizer_dir, 
    vocab_size, 
    special_tokens_num=SPECIAL_TOKENS_NUM
):
    tokenizer = Tokenizer(models.BPE()) #创建一个 BPE 分词器。
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    #设置 ByteLevel 预分词器。ByteLevel 的作用是把文本映射到字节级表示，
    #使任意 UTF-8 文本原则上都能够被编码，不容易出现真正无法识别的字符。（可见字符映射）
    
    special_tokens_list = [ #特殊 Token
        "<|endoftext|>", "<|im_start|>", "<|im_end|>", 
        "<|object_ref_start|>", "<|object_ref_end|>", "<|box_start|>", "<|box_end|>", "<|quad_start|>", "<|quad_end|>", 
        "<|vision_start|>", "<|vision_end|>", "<|vision_pad|>", "<|image_pad|>", "<|video_pad|>", 
        "<|audio_start|>", "<|audio_end|>", "<|audio_pad|>", "<tts_pad>", "<tts_text_bos>", "<tts_text_eod>", "<tts_text_bos_single>"
    ]
    
    additional_tokens_list = [ #功能标记
        "<tool_call>", "</tool_call>",#助手发起工具调用
        "<tool_response>", "</tool_response>", #工具返回结果
        "<think>", "</think>" #模型的思考内容
    ]
    num_buffer = special_tokens_num - len(special_tokens_list + additional_tokens_list)
    #计算还剩多少特殊 Token 位置
    buffer_tokens = [f"<|buffer{i}|>" for i in range(1, num_buffer + 1)] 
    # 预留一定数量的token位置，这些位置预留给未来扩展。预先保留 Token ID，可以减少未来新增 Token 时破坏兼容性的风险。
    all_special_tokens = special_tokens_list + additional_tokens_list + buffer_tokens
    #组合全部保留 Token。

    #创建 BPE 训练器：
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size, #最终尽量生成vocab_size个 Token。
        show_progress=True,    #显示训练进度
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),#保证初始字节字符集合完整
        special_tokens=all_special_tokens #保证指定 Token 完整进入词表，不会被拆开
    )
    texts = get_texts(data_path)#创建文本生成器，此时还没有真正把所有数据读入内存。
    tokenizer.train_from_iterator(texts, trainer=trainer)
    #开始真正训练：读取语料；统计字节或子串的共现频率；反复合并高频组合（merge）；
    # 建立 Token 与 Token ID 的映射（ids）；生成目标词表。
    tokenizer.decoder = decoders.ByteLevel()
    #设置对应的 ByteLevel 解码器，这个解码器可 让字节级 Token 能重新还原为 UTF-8 文本再转换为人类可读文本
    tokenizer.add_special_tokens(special_tokens_list)
    #确保核心特殊 Token 被登记为真正的 special token。

    # 保存并调整 tokenizer.json
    os.makedirs(tokenizer_dir, exist_ok=True) #创建输出目录
    tokenizer.save(os.path.join(tokenizer_dir, "tokenizer.json"))
    #保存完整分词器：model_learn_tokenizer/tokenizer.json
    tokenizer.model.save(tokenizer_dir)
    #另外保存 BPE 模型的独立文件，一般包括：vocab.json；merges.txt
    tokenizer_json_path = os.path.join(tokenizer_dir, "tokenizer.json")
    #保存 tokenizer.json 的完整路径，方便接下来重新读取。
    #把刚保存的 JSON 文件读回 Python：
    with open(tokenizer_json_path, 'r', encoding='utf-8') as f:
        tokenizer_data = json.load(f)
    #遍历 Added Tokens 的配置：
    for token_info in tokenizer_data.get('added_tokens', []):
        if token_info['content'] not in special_tokens_list:
            token_info['special'] = False
            #只有 special_tokens_list 中的核心 Token 被标记为真正的 special token。
    #重新写回 JSON：
    with open(tokenizer_json_path, 'w', encoding='utf-8') as f:
        json.dump(tokenizer_data, f, ensure_ascii=False, indent=2)
    
    #生成 Added Token 配置
    added_tokens_decoder = {} #创建一个字典，记录 Token ID 到 Token 配置的映射。
    for i, token in enumerate(all_special_tokens): #遍历全部保留 Token
        idx = tokenizer.token_to_id(token) #查询 Token 的真实整数 ID
        added_tokens_decoder[str(idx)] = { #创建特殊token的json文件
            "content": token,
            "lstrip": False,
            "normalized": False,
            "rstrip": False,
            "single_word": False,
            "special": True if token in special_tokens_list else False
        }

    #生成 tokenizer_config.json：
    config = {
        "add_bos_token": False,
        "add_eos_token": False,
        "add_prefix_space": False,
        "added_tokens_decoder": added_tokens_decoder,
        "additional_special_tokens": [t for t in special_tokens_list if t not in ["<|endoftext|>"]],
        "bos_token": "<|im_start|>",
        "clean_up_tokenization_spaces": False,
        "eos_token": "<|im_end|>",
        "legacy": True,
        "model_max_length": 131072,
        "pad_token": "<|endoftext|>",
        "sp_model_kwargs": {},
        "spaces_between_special_tokens": False,
        "unk_token": "<|endoftext|>",
        "image_token": "<|image_pad|>",
        "audio_token": "<|audio_pad|>",
        "video_token": "<|video_pad|>",
        "vision_bos_token": "<|vision_start|>",
        "vision_eos_token": "<|vision_end|>",
        "audio_bos_token": "<|audio_start|>",
        "audio_eos_token": "<|audio_end|>",
        "chat_template": "{%- if tools %}\n    {{- '<|im_start|>system\\n' }}\n    {%- if messages[0].role == 'system' %}\n        {{- messages[0].content + '\\n\\n' }}\n    {%- endif %}\n    {{- \"# Tools\\n\\nYou may call one or more functions to assist with the user query.\\n\\nYou are provided with function signatures within <tools></tools> XML tags:\\n<tools>\" }}\n    {%- for tool in tools %}\n        {{- \"\\n\" }}\n        {{- tool | tojson }}\n    {%- endfor %}\n    {{- \"\\n</tools>\\n\\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\\n<tool_call>\\n{\\\"name\\\": <function-name>, \\\"arguments\\\": <args-json-object>}\\n</tool_call><|im_end|>\\n\" }}\n{%- else %}\n    {%- if messages[0].role == 'system' %}\n        {{- '<|im_start|>system\\n' + messages[0].content + '<|im_end|>\\n' }}\n    {%- endif %}\n{%- endif %}\n{%- set ns = namespace(multi_step_tool=true, last_query_index=messages|length - 1) %}\n{%- for message in messages[::-1] %}\n    {%- set index = (messages|length - 1) - loop.index0 %}\n    {%- if ns.multi_step_tool and message.role == \"user\" and message.content is string and not(message.content.startswith('<tool_response>') and message.content.endswith('</tool_response>')) %}\n        {%- set ns.multi_step_tool = false %}\n        {%- set ns.last_query_index = index %}\n    {%- endif %}\n{%- endfor %}\n{%- for message in messages %}\n    {%- if message.content is string %}\n        {%- set content = message.content %}\n    {%- else %}\n        {%- set content = '' %}\n    {%- endif %}\n    {%- if (message.role == \"user\") or (message.role == \"system\" and not loop.first) %}\ n        {{- '<|im_start|>' + message.role + '\\ n' + content + '<|im_end|>' + '\\ n' }}\ n    {%- elif message.role == \"assistant\" %}\ n        {%- set reasoning_content = '' %}\ n        {%- if message.reasoning_content is string %}\ n            {%- set reasoning_content = message.reasoning_content %}\ n        {%- else %}\ n            {%- if '</think>' in content %}\ n                {%- set reasoning_content = content.split('</think>')[0].rstrip('\\ n').split('<think>')[-1].lstrip('\\ n') %}\ n                {%- set content = content.split('</think>')[-1].lstrip('\\ n') %}\ n            {%- endif %}\ n        {%- endif %}\ n        {%- if true %}\ n            {{- '<|im_start|>' + message.role + '\\ n<think>\\ n' + reasoning_content.strip('\\ n') + '\\ n</think>\\ n\\ n' + content.lstrip('\\ n') }}\ n        {%- endif %}\ n        {%- if message.tool_calls %}\ n            {%- for tool_call in message.tool_calls %}\ n                {%- if (loop.first and content) or (not loop.first) %}\ n                    {{- '\\ n' }}\ n                {%- endif %}\ n                {%- if tool_call.function %}\ n                    {%- set tool_call = tool_call.function %}\ n                {%- endif %}\ n                {{- '<tool_call>\\ n{\"name\": \"' }}\ n                {{- tool_call.name }}\ n                {{- '\", \"arguments\": ' }}\ n                {%- if tool_call.arguments is string %}
                    {{- tool_call.arguments }}
                {%- else %}
                    {{- tool_call.arguments | tojson }}
                {%- endif %}
                {{- '}\\_n</tool_call>' }}
            {%- endfor %}
        {%- endif %}
        {{- '<|im_end|>\\_n' }}
    {%- elif message.role == \"tool\" %}
        {:- if loop.first or (messages[loop.index0 - 1].role != \"tool\") %}
            {{- '<|im_start|>user' }}
        {:endif %}
        {{- '\\_n<tool_response>\\_n' }}
        {{- content }}
        {{- '\\_n</tool_response>' }}
        {:if loop.last or (messages[loop.index0 + 1].role != \"tool\") %}
            {{- '<|im_end|>\\_n' }}
        {:endif %}
    {:endif %}
{:endfor %}
{:if add_generation_prompt %}
    {{- '<|im_start|>assistant\\_n' }}
    {:if open_thinking is defined and open_thinking is true %}
        {{- '<think>\\_n' }}
    {:else %}
        {{- '<think>\\_n\\_n</think>\\_n\\_n' }}
    {:endif %}
{:endif }",
        #chat_template：把结构化消息转换成模型真正看到的文本
        "tokenizer_class": "PreTrainedTokenizerFast"
    }

    #把配置保存成：model_learn_tokenizer/tokenizer_config.json
    with open(os.path.join(tokenizer_dir, "tokenizer_config.json"), "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=4)
    print("Tokenizer training completed.")# 提示训练及保存完成


#### 4 测试分词器

In [30]:

def eval_tokenizer(tokenizer_dir):# 定义分词器评估函数。
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir)
    # 使用 Transformers 加载刚才自己训练并保存的本地分词器（存储地址：tokenizer_dir）。

    messages = [ #构造一段 system/user/assistant 多轮对话，用于测试聊天模板。
        {"role": "system", "content": "你是一个优秀的聊天机器人，总是给我正确的回应！"},
        {"role": "user", "content": '你来自哪里？'},
        {"role": "assistant", "content": '我来自月球'},
        {"role": "user", "content": '你到底来自哪里？'},
        {"role": "assistant", "content": '我来自地球'}
    ]

    #应用 chat_template 把结构化消息messages转换成模型真正看到的文本。
    # tokenize=False 表示现在只生成格式化字符串，还不转换为 Token ID。
    new_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )
    print('-'*100)
    print(new_prompt) #显示模板处理后的完整文本
    print('-'*100)
    print('tokenizer词表长度：', len(tokenizer))
    model_inputs = tokenizer(new_prompt)
    print('encoder长度：', len(model_inputs['input_ids']))# 打印该 prompt 被切成多少个 Token。
    response = tokenizer.decode(model_inputs['input_ids'], skip_special_tokens=False)
    #把 Token ID 解码回来
    print('decoder一致性：', response == new_prompt, "\n")
    print('-'*100)
    print('压缩率测试（Chars/Tokens）：')
    test_texts = [
        # 中文样本 (约200字)
        "人工智能是计算机科学的一个分支，它企图了解智能的实质，并生产出一种新的能以人类智能相似的方式做出反应的智能机器，该领域的研究包括机器人、语言识别、图像识别、自然语言处理和专家系统等。人工智能从诞生以来，理论和技术日益成熟，应用领域也不断扩大，可以设想，未来人工智能带来的科技产品，将会是人类智慧的“容器”。人工智能可以对人的意识、思维的信息过程的模拟。人工智能不是人的智能，但能像人那样思考、也可能超过人的智能。",
        "星际航行是指在星系内甚至星系间的空间中进行的航行。由于宇宙空间极其广阔，传统的化学火箭动力在恒星间航行时显得力不从心。科学家们提出了多种方案，包括离子推进器、核热火箭、甚至是利用反物质作为能源的设想。此外，曲率驱动和虫洞旅行等科幻概念也在理论物理研究中被反复探讨。尽管目前人类的足迹仅限于月球，但随着核聚变技术和材料科学的突破，前往火星乃至更遥远的太阳系边缘将成为可能。",
        # 英文样本 (约200词/字符)
        "Large language models (LLMs) are a type of artificial intelligence (AI) trained on vast amounts of text data to understand and generate human-like language. These models use deep learning techniques, specifically transformers, to process and predict the next word in a sequence. LLMs like GPT-4, Llama, and Claude have demonstrated remarkable capabilities in coding, translation, and creative writing. However, they also face challenges such as hallucinations, where the model generates factually incorrect information, and the need for significant computational resources.",
        "The development of sustainable energy is crucial for the future of our planet. As climate change continues to impact global weather patterns, transitioning from fossil fuels to renewable sources like solar, wind, and hydroelectric power has become an urgent priority. Innovations in battery storage technology and smart grid management are essential to ensure a reliable energy supply. International cooperation and policy frameworks are also necessary to drive the global shift towards a greener economy and reduce carbon emissions.",
        # 混合样本
        "Python 是一种高级编程语言，以其简洁的语法和强大的生态系统而闻名。It is widely used in data science, machine learning, and web development. 开发者可以利用 NumPy, Pandas, and PyTorch 等库快速构建复杂的应用。学习 Python 的过程非常愉快，因为它的代码读起来就像英语一样。Whether you are a beginner or an expert, Python offers something for everyone.",
    ]
    
    total_compression = 0
    for i, text in enumerate(test_texts):
        encoded = tokenizer.encode(text)
        token_count = len(encoded)
        char_count = len(text)
        compression_ratio = char_count / token_count
        total_compression += compression_ratio
        print(f"样本 {i+1} | 字符数: {char_count:4} | Tokens: {token_count:3} | 压缩率: {compression_ratio:.2f}")
    
    print(f"平均压缩率: {total_compression / len(test_texts):.2f}")
    print('-'*100)
    print('流式解码（字节缓冲）测试：')
    input_ids = model_inputs['input_ids']
    token_cache = []
    for tid in input_ids:
        token_cache.append(tid)
        current_decode = tokenizer.decode(token_cache)
        if current_decode and '\ufffd' not in current_decode:
            display_ids = token_cache[0] if len(token_cache) == 1 else token_cache
            raw_tokens = [tokenizer.convert_ids_to_tokens(int(t)) for t in (token_cache if isinstance(token_cache, list) else [token_cache])]
            print(f'Token ID: {str(display_ids):15} -> Raw: {str(raw_tokens):20} -> Decode Str: {current_decode}')
            token_cache = []

# 程序入口，只有直接执行这个文件时，下面两行才运行。
if __name__ == '__main__':
    train_tokenizer(DATA_PATH, TOKENIZER_DIR, VOCAB_SIZE)
    eval_tokenizer(TOKENIZER_DIR)


NameError: name 'train_tokenizer' is not defined

加载minimind现成分词器：

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "./model",
    local_files_only=True
)

In [32]:
text = "你好，MiniMind！"

ids = tokenizer.encode(text)
tokens = tokenizer.convert_ids_to_tokens(ids)
decoded = tokenizer.decode(ids)

print(ids)
print(tokens)
print(decoded)

[1968, 294, 80, 301, 108, 80, 916, 1364]
['ä½łå¥½', 'ï¼Į', 'M', 'in', 'i', 'M', 'ind', 'ï¼ģ']
你好，MiniMind！


补充说明以上函数中的
```model_inputs = tokenizer(new_prompt)```
这一步不仅“切分 Token”，还完成了从字符串到模型输入数据的一整套转换。

大致经过：
```text
new_prompt 字符串
→ 规范化
→ ByteLevel 预处理
→ BPE 分词
→ 查词表
→ 转换成 Token ID
→ 生成 attention_mask
→ 保存到 model_inputs
```

等价于调用：
```model_inputs = tokenizer.__call__(new_prompt)```

```python
    new_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False
        )
```
是已经经过聊天模板chat_template处理的字符串

比如：
```text
<|im_start|>user
你好<|im_end|>
<|im_start|>assistant
<think>

</think>

你好！<|im_end|>
```
它目前仍然是普通 Python 字符串：print(type(new_prompt))-> <class 'str'>

model_inputs 通常类似字典：
```python
{
    "input_ids": [...],
    "attention_mask": [...]
}
```

其中，

**model_inputs["input_ids"]**
这就是分词后得到的 Token ID：例如：[1, 1092, 198, 1968, 2, 1, 1536, 198, ...]

**model_inputs["attention_mask"]**

它告诉模型哪些位置是真实内容，哪些是 Padding：
```text
1 → 真实 Token，应该参与注意力
0 → PAD，应该忽略
```
如果没有要求补齐，所有位置通常都是 1：
[1, 1, 1, 1, 1, 1]

In [33]:
    messages = [
        {"role": "system", "content": "你是一个优秀的聊天机器人，总是给我正确的回应！"},
        {"role": "user", "content": '你来自哪里？'},
        {"role": "assistant", "content": '我来自月球'},
        {"role": "user", "content": '你到底来自哪里？'},
        {"role": "assistant", "content": '我来自地球'}
    ]
    new_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )

In [34]:
print("原始 Prompt：")
print(new_prompt)

tokens = tokenizer.tokenize(new_prompt)
print("\n分词结果：")
print(tokens)

token_ids = tokenizer.convert_tokens_to_ids(tokens)
print("\nToken ID：")
print(token_ids)

model_inputs = tokenizer(new_prompt)
print("\n最终模型输入：")
print(model_inputs)

原始 Prompt：
<|im_start|>system
你是一个优秀的聊天机器人，总是给我正确的回应！<|im_end|>
<|im_start|>user
你来自哪里？<|im_end|>
<|im_start|>assistant
<think>

</think>

我来自月球<|im_end|>
<|im_start|>user
你到底来自哪里？<|im_end|>
<|im_start|>assistant
<think>

</think>

我来自地球<|im_end|>


分词结果：
['<|im_start|>', 's', 'ystem', 'Ċ', 'ä½ł', 'æĺ¯ä¸Ģä¸ª', 'ä¼ĺ', 'ç§Ģ', 'çļĦ', 'èģĬ', 'å¤©', 'æľºåĻ¨äºº', 'ï¼Į', 'æĢ»æĺ¯', 'ç»ĻæĪĳ', 'æŃ£ç¡®çļĦ', 'åĽŀ', 'åºĶ', 'ï¼ģ', '<|im_end|>', 'Ċ', '<|im_start|>', 'us', 'er', 'Ċ', 'ä½ł', 'æĿ¥èĩª', 'åĵª', 'éĩĮ', 'ï¼Ł', '<|im_end|>', 'Ċ', '<|im_start|>', 'ass', 'ist', 'ant', 'Ċ', '<think>', 'Ċ', 'Ċ', '</think>', 'Ċ', 'Ċ', 'æĪĳ', 'æĿ¥èĩª', 'æľĪ', 'çĲĥ', '<|im_end|>', 'Ċ', '<|im_start|>', 'us', 'er', 'Ċ', 'ä½ł', 'åĪ°', 'åºķ', 'æĿ¥èĩª', 'åĵª', 'éĩĮ', 'ï¼Ł', '<|im_end|>', 'Ċ', '<|im_start|>', 'ass', 'ist', 'ant', 'Ċ', '<think>', 'Ċ', 'Ċ', '</think>', 'Ċ', 'Ċ', 'æĪĳ', 'æĿ¥èĩª', 'åľ°çĲĥ', '<|im_end|>', 'Ċ']

Token ID：
[1, 118, 4849, 234, 441, 1001, 985, 5436, 296, 5358, 993, 3697, 294, 6056, 3530, 5154, 10

```text
原始对话 messages
        ↓
tokenizer_config.json 的 chat_template
        ↓
带角色标记的字符串 prompt
        ↓
tokenizer.json
        ↓
Token ID
        ↓
MiniMind 模型
        ↓
预测下一个 Token ID
        ↓
tokenizer.json
        ↓
输出文字
```